# LeetCode 227 — Basic Calculator II bằng Python

Link đề: https://leetcode.com/problems/basic-calculator-ii/ · Độ khó: **Medium** · Chủ đề: String, Stack, Math (tính biểu thức)

Skeleton LeetCode:

```python
class Solution:
    def calculate(self, s: str) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc. Đề **cấm** dùng `eval()` để nộp bài; trong notebook ta chỉ dùng `eval` như **công cụ đối chiếu khi test** (sau khi đổi `/` thành `//` cho khớp phép chia nguyên với toán hạng không âm), còn mọi lời giải đều tự cài đặt.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính giá trị biểu thức chỉ gồm số nguyên **không âm** và `+ - * /` (có thể có dấu cách), với **phép chia nguyên cắt về `0`** và **độ ưu tiên**: `*`, `/` tính trước `+`, `-`.
2. Hiểu cách **đọc số nhiều chữ số** từ chuỗi (`num = num * 10 + chữ số`) và cách **bỏ qua khoảng trắng**.
3. Đi từ cách **tách token + hai lượt (nhân/chia trước, cộng/trừ sau)** → **một lượt với stack** (đẩy số kèm dấu, nhân/chia với phần tử đỉnh) → **một lượt, `O(1)` bộ nhớ** (chỉ giữ `last` và `total`) → **văn phạm đệ quy** `expr → term (+/- term)*`.
4. Phân biệt **chia nguyên cắt về `0`** (`int(a / b)`) với **chia sàn** (`a // b`) khi một vế **âm** — nguồn lỗi kinh điển của bài này.
5. Biết dùng `eval` làm **oracle kiểm thử** (không dùng khi nộp), và tránh các lỗi: bỏ qua độ ưu tiên, quên số cuối, đọc từng chữ số riêng lẻ, xử lý sai khoảng trắng, chia sàn với số âm.

## 2. Đề bài & ví dụ

Cho chuỗi `s` biểu diễn một **biểu thức**, hãy **tính giá trị** biểu thức và trả về.

- **Phép chia nguyên** phải **cắt về `0`** (truncate toward zero).
- Có thể giả định biểu thức **luôn hợp lệ**. Mọi kết quả trung gian nằm trong khoảng `[-2^31, 2^31 - 1]`.
- **Không được** dùng hàm có sẵn nào tính chuỗi thành biểu thức toán học, như `eval()`.

**Ví dụ 1**

- Input: `s = "3+2*2"`
- Output: `7`

**Ví dụ 2**

- Input: `s = " 3/2 "`
- Output: `1`

**Ví dụ 3**

- Input: `s = " 3+5 / 2 "`
- Output: `5`

**Ràng buộc:** `1 <= s.length <= 3 * 10^5`; `s` gồm các số nguyên và các toán tử `'+'`, `'-'`, `'*'`, `'/'` cách nhau bởi một số khoảng trắng; `s` là biểu thức hợp lệ; mọi số nguyên là số **không âm** trong `[0, 2^31 - 1]`; đáp án được đảm bảo nằm trong số nguyên 32-bit.

In [1]:
from typing import List
import random
import time

examples = [
    ("3+2*2", 7),
    (" 3/2 ", 1),
    (" 3+5 / 2 ", 5),
]
for s, expected in examples:
    print(f"s={s!r} -> kỳ vọng {expected}")

s='3+2*2' -> kỳ vọng 7
s=' 3/2 ' -> kỳ vọng 1
s=' 3+5 / 2 ' -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi, `isdigit()` và bỏ khoảng trắng

`for ch in s` duyệt từng ký tự; `ch.isdigit()` cho biết ký tự có phải chữ số `'0'..'9'` không; `ch == ' '` là khoảng trắng. Một toán tử là ký tự thuộc `'+-*/'`. Quy tắc phân loại: **chữ số** → gom vào số; **khoảng trắng** → bỏ qua; còn lại → **toán tử**.

In [2]:
s = " 3+5 / 2 "
for i, ch in enumerate(s):
    kind = "chữ số" if ch.isdigit() else ("khoảng trắng" if ch == " " else "toán tử")
    print(f"i={i}: {ch!r:>4} -> {kind}")

i=0:  ' ' -> khoảng trắng
i=1:  '3' -> chữ số
i=2:  '+' -> toán tử
i=3:  '5' -> chữ số
i=4:  ' ' -> khoảng trắng
i=5:  '/' -> toán tử
i=6:  ' ' -> khoảng trắng
i=7:  '2' -> chữ số
i=8:  ' ' -> khoảng trắng


### 3.2. Ghép số nhiều chữ số: `num = num * 10 + int(ch)`

Chuỗi `"123"` có ba chữ số liên tiếp. Đọc từ trái sang phải: bắt đầu `num = 0`; gặp `'1'` → `num = 0*10 + 1 = 1`; gặp `'2'` → `num = 1*10 + 2 = 12`; gặp `'3'` → `num = 12*10 + 3 = 123`. Gặp **toán tử** (hoặc hết chuỗi) thì số đã xong, ta dùng nó rồi **đặt lại `num = 0`**.

In [3]:
num = 0
for ch in "123":
    num = num * 10 + int(ch)
    print(f"gặp {ch!r} -> num = {num}")
print(num + 1)      # 124: là số nguyên thật, dùng để tính toán được

gặp '1' -> num = 1
gặp '2' -> num = 12
gặp '3' -> num = 123
124


### 3.3. Độ ưu tiên toán tử và "biểu thức là tổng các số hạng"

Vì `*` và `/` ưu tiên hơn `+` và `-`, mọi biểu thức đều có thể xem là **tổng (có dấu) của các số hạng (term)**, mỗi số hạng là một chuỗi **nhân/chia** liên tiếp. Ví dụ `3+5/2*4-1` có ba số hạng: `+3`, `+(5/2*4)` và `-1`. Bài toán thành: **tính từng số hạng (nhân/chia trái sang phải), rồi cộng lại với dấu**.

In [4]:
print(3 + 5 // 2 * 4 - 1)          # 10: Python cũng ưu tiên * và / (// ở đây) trước + và -
terms = [3, 5 // 2 * 4, -1]
print(terms, sum(terms))           # [3, 8, -1] 10

10
[3, 8, -1] 10


### 3.4. Stack (ngăn xếp) bằng `list`

**Stack**: vào sau – ra trước. Trong Python dùng `list`: `append(x)` đẩy vào đỉnh, `pop()` lấy khỏi đỉnh, `stack[-1]` xem đỉnh. Ta sẽ đẩy **các số hạng đã có dấu** vào stack; với `*` hoặc `/`, lấy số hạng **đỉnh** ra, tính tiếp với số mới rồi đẩy kết quả lại.

In [5]:
stack = []
stack.append(3)
stack.append(-2)
print(stack, stack[-1])             # [3, -2] -2
top = stack.pop()
stack.append(top * 5)               # nhân số hạng ở đỉnh với 5
print(stack, sum(stack))            # [3, -10] -7

[3, -2] -2
[3, -10] -7


### 3.5. Chia nguyên cắt về `0` khác chia sàn `//`

- `//` là **chia sàn** (làm tròn xuống, về `-∞`): `7 // 2 = 3` nhưng `-7 // 2 = -4`.
- Đề yêu cầu **cắt về 0** (truncate toward zero): `-7 / 2 → -3`.
- Trong Python: `int(a / b)` cắt về `0` (với số trong phạm vi đề, phép chia thực đủ chính xác); hoặc `math.trunc(a / b)`.

Với `a`, `b` **không âm** thì `a // b == int(a / b)`. Chỉ khác nhau khi có số **âm** — và bài này có thể có số hạng âm trong stack (ví dụ số hạng `-5` trong `1-5/2`).

In [6]:
from math import trunc
for a, b in [(7, 2), (-7, 2), (7, -2), (-7, -2)]:
    print(f"a={a:>2}, b={b:>2}: a // b = {a // b:>2}   int(a / b) = {int(a / b):>2}   trunc = {trunc(a / b):>2}")

a= 7, b= 2: a // b =  3   int(a / b) =  3   trunc =  3
a=-7, b= 2: a // b = -4   int(a / b) = -3   trunc = -3
a= 7, b=-2: a // b = -4   int(a / b) = -3   trunc = -3
a=-7, b=-2: a // b =  3   int(a / b) =  3   trunc =  3


### 3.6. Dùng "dấu hiệu cuối" (sentinel) hoặc kiểm tra `i == n - 1`

Một số chỉ được xử lý khi ta gặp **toán tử kế tiếp**. Số cuối cùng không có toán tử phía sau nên dễ bị quên. Hai cách: (a) nối thêm một toán tử giả `'+'` vào cuối chuỗi; (b) xử lý khi `i` là chỉ số cuối. Cả hai đều sẽ được dùng.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính tay `" 3+5 / 2 "`:

1. Bỏ khoảng trắng: `3+5/2`.
2. `/` ưu tiên hơn `+`: tính `5 / 2 = 2` (cắt về 0) trước.
3. Cộng: `3 + 2 = 5`.

Nếu viết theo số hạng: `+3` và `+(5/2) = +2` → `3 + 2 = 5`.

Tính tay `"3+2*2"`: số hạng `+3` và `+(2*2) = +4` → `7`. Nếu tính **trái sang phải bỏ qua độ ưu tiên** ta được `(3+2)*2 = 10` — sai.

**Cách nhìn then chốt:** duyệt biểu thức từ trái sang phải. Khi gặp một **dấu mới**, ta mới biết số vừa đọc đã **hoàn chỉnh** và nó thuộc về phép toán **trước đó**: nếu dấu trước đó là `+`/`-` thì số đó **bắt đầu một số hạng mới**; nếu là `*`/`/` thì số đó **nhân/chia vào số hạng đang dở**. Nên ta cần nhớ **"dấu đứng trước số"**.

## 5. Cách 1 — Brute Force / baseline: tách token rồi tính hai lượt

**Ý tưởng.**

1. **Tách token**: duyệt chuỗi, gom chữ số thành số nguyên, bỏ khoảng trắng, giữ các toán tử → danh sách như `[3, '+', 5, '/', 2]`.
2. **Lượt 1 (ưu tiên cao):** quét từ trái sang phải; với `*`, `/` thì tính ngay vào phần tử **cuối** của danh sách kết quả tạm; với `+`, `-` thì **chuyển nguyên** (giữ toán tử và số) sang danh sách tạm.
3. **Lượt 2:** cộng/trừ từ trái sang phải.

Vì ở lượt 1 ta giữ dấu `+`/`-` **riêng** khỏi số (số luôn không âm), dùng `//` là an toàn.
#### Code demo

In [7]:
def tokenize(s):
    tokens, i, n = [], 0, len(s)
    while i < n:
        if s[i] == " ":
            i += 1
        elif s[i].isdigit():
            j = i
            while j < n and s[j].isdigit():          # gom mọi chữ số liên tiếp
                j += 1
            tokens.append(int(s[i:j]))
            i = j
        else:
            tokens.append(s[i])                     # toán tử
            i += 1
    return tokens


def calculate_two_pass(s):
    tokens = tokenize(s)
    # Lượt 1: gộp * và /
    reduced = [tokens[0]]
    for k in range(1, len(tokens), 2):
        op, value = tokens[k], tokens[k + 1]
        if op == "*":
            reduced[-1] = reduced[-1] * value
        elif op == "/":
            reduced[-1] = reduced[-1] // value      # an toàn: reduced[-1] không âm vì dấu +/- để riêng
        else:
            reduced.append(op)
            reduced.append(value)
    # Lượt 2: cộng/trừ
    result = reduced[0]
    for k in range(1, len(reduced), 2):
        result += reduced[k + 1] if reduced[k] == "+" else -reduced[k + 1]
    return result


print(tokenize(" 3+5 / 2 "))
for s, expected in examples:
    print(repr(s), "->", calculate_two_pass(s), "(kỳ vọng", expected, ")")

[3, '+', 5, '/', 2]
'3+2*2' -> 7 (kỳ vọng 7 )
' 3/2 ' -> 1 (kỳ vọng 1 )
' 3+5 / 2 ' -> 5 (kỳ vọng 5 )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Mỗi ký tự được đọc một lần khi tách token và mỗi token một lần ở mỗi lượt: **Time `O(n)`**; danh sách token và danh sách tạm tốn **Space `O(n)`**.

**Nút thắt.** Ta tạo hai danh sách trung gian chỉ để tính. Ta có thể **gộp** tách token và tính vào **một** lượt, nhờ việc nhớ "dấu đứng trước số".

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt từng ký tự, giữ ba biến:

- `num`: số đang đọc dở (ban đầu `0`);
- `op`: **dấu đứng trước** `num` (ban đầu `'+'`, như thể biểu thức bắt đầu bằng `+`);
- (stack hoặc `last`/`total`): kết quả các số hạng.

Khi gặp **toán tử mới** (hoặc hết chuỗi), số `num` đã xong; ta xử lý nó theo `op` cũ:

| `op` cũ | hành động |
|---|---|
| `+` | **bắt đầu** số hạng mới `+num` |
| `-` | **bắt đầu** số hạng mới `-num` |
| `*` | số hạng đang dở `*= num` |
| `/` | số hạng đang dở `= cắt về 0 (số hạng / num)` |

Sau đó `op = toán tử mới` và `num = 0`. Cuối cùng **cộng mọi số hạng**. Đây là **một lượt `O(n)`**: stack chỉ giữ các số hạng, và thật ra ta chỉ cần nhớ **số hạng cuối** (`last`) cùng **tổng các số hạng đã xong** (`total`) để đạt `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

### Cách 2 — Một lượt với stack

**Giải thích.** Mỗi số hạng được đẩy vào `stack` **kèm dấu**: `+num` hoặc `-num`. Với `*`, `/` lấy phần tử đỉnh `stack.pop()` ra tính với `num` rồi đẩy kết quả lại. Cuối cùng `sum(stack)`. Ở bản này dùng **dấu hiệu cuối** `+` để số cuối cũng được xử lý.
#### Code demo

In [8]:
def calculate_stack(s):
    stack = []
    num = 0
    op = "+"                                  # dấu đứng trước num (như biểu thức bắt đầu bằng +)
    for ch in s + "+":                        # nối thêm '+' để xử lý số cuối cùng
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch != " ":                       # ch là toán tử: num đã hoàn chỉnh
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:                             # op == "/"
                stack.append(int(stack.pop() / num))   # cắt về 0, KHÔNG dùng // vì đỉnh có thể âm
            op = ch
            num = 0
    return sum(stack)


for s, expected in examples:
    print(repr(s), "->", calculate_stack(s), "(kỳ vọng", expected, ")")

'3+2*2' -> 7 (kỳ vọng 7 )
' 3/2 ' -> 1 (kỳ vọng 1 )
' 3+5 / 2 ' -> 5 (kỳ vọng 5 )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def calculate(self, s: str) -> int:
        stack = []
        num = 0
        op = "+"
        for ch in s + "+":
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch != " ":
                if op == "+":
                    stack.append(num)
                elif op == "-":
                    stack.append(-num)
                elif op == "*":
                    stack.append(stack.pop() * num)
                else:
                    stack.append(int(stack.pop() / num))
                op = ch
                num = 0
        return sum(stack)


print(Solution().calculate("3+2*2"))     # 7

7


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (stack có thể chứa tới ~`n/2` số hạng, ví dụ `1+1+1+...`).

### Cách 3 — Một lượt, `O(1)` bộ nhớ: chỉ giữ `last` và `total`

**Giải thích.** Stack chỉ có **phần tử đỉnh** bị sửa bởi `*`, `/`; các phần tử bên dưới không bao giờ đụng tới nữa nên chỉ cần **tổng** của chúng. Ta giữ:

- `last`: số hạng đang dở (phần tử đỉnh stack);
- `total`: tổng các số hạng **đã chốt** (mọi phần tử dưới đỉnh).

Với `op = '+'`/`'-'`: chốt `last` (`total += last`) rồi bắt đầu số hạng mới `last = ±num`. Với `*`, `/`: sửa `last`. Cuối cùng trả `total + last`. Ở bản này dùng điều kiện `i == n - 1` thay cho nối thêm ký tự.
#### Code demo

In [10]:
def calculate_constant_space(s):
    total, last = 0, 0
    num = 0
    op = "+"
    n = len(s)
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if (not ch.isdigit() and ch != " ") or i == n - 1:      # gặp toán tử, hoặc đã tới ký tự cuối
            if op == "+":
                total += last
                last = num
            elif op == "-":
                total += last
                last = -num
            elif op == "*":
                last = last * num
            else:
                last = int(last / num)
            op = ch
            num = 0
    return total + last


for s, expected in examples:
    print(repr(s), "->", calculate_constant_space(s), "(kỳ vọng", expected, ")")

'3+2*2' -> 7 (kỳ vọng 7 )
' 3/2 ' -> 1 (kỳ vọng 1 )
' 3+5 / 2 ' -> 5 (kỳ vọng 5 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (chỉ vài biến số nguyên).

### Cách 4 — Văn phạm đệ quy: `expr → term (('+'|'-') term)*`, `term → số (('*'|'/') số)*`

**Giải thích.** Ghi quy tắc độ ưu tiên thành **văn phạm**: một **biểu thức** là một dãy **số hạng** nối bằng `+`/`-`; một **số hạng** là một dãy **số** nối bằng `*`/`/`. Mỗi quy tắc là một hàm; hàm `term` tính xong nhân/chia **trước** khi trả về cho `expr`, nên độ ưu tiên tự đúng. Cách này mở rộng tự nhiên sang **dấu ngoặc** (Basic Calculator I và III).
#### Code demo

In [11]:
def calculate_grammar(s):
    tokens = tokenize(s)
    pos = 0

    def number():
        nonlocal pos
        value = tokens[pos]
        pos += 1
        return value

    def term():
        nonlocal pos
        value = number()
        while pos < len(tokens) and tokens[pos] in ("*", "/"):
            op = tokens[pos]
            pos += 1
            rhs = number()
            value = value * rhs if op == "*" else value // rhs     # value không âm trong một term
        return value

    def expr():
        nonlocal pos
        value = term()
        while pos < len(tokens) and tokens[pos] in ("+", "-"):
            op = tokens[pos]
            pos += 1
            rhs = term()
            value = value + rhs if op == "+" else value - rhs
        return value

    return expr()


for s, expected in examples:
    print(repr(s), "->", calculate_grammar(s), "(kỳ vọng", expected, ")")

'3+2*2' -> 7 (kỳ vọng 7 )
' 3/2 ' -> 1 (kỳ vọng 1 )
' 3+5 / 2 ' -> 5 (kỳ vọng 5 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (danh sách token; độ sâu đệ quy chỉ 2 tầng vì không có ngoặc).

> Cách 3 là bản gọn và tối ưu bộ nhớ; Cách 2 là bản "sách giáo khoa" dễ giải thích nhất (nhớ cả hai). Cách 4 là nền tảng để làm các bài có ngoặc.

## 8. TRACE từng bước

Ta trace Cách 2 (stack) trên `" 3+5 / 2 "` và Cách 3 (`last`/`total`) trên `"3+2*2-4/3"`.

In [12]:
def trace_stack(s):
    print(f"s = {s!r}")
    stack, num, op = [], 0, "+"
    for i, ch in enumerate(s + "+"):
        note = ""
        if ch.isdigit():
            num = num * 10 + int(ch)
            note = f"chữ số -> num={num}"
        elif ch == " ":
            note = "bỏ qua khoảng trắng"
        else:
            if op == "+":
                stack.append(num)
                note = f"op cũ '+': đẩy {num}"
            elif op == "-":
                stack.append(-num)
                note = f"op cũ '-': đẩy {-num}"
            elif op == "*":
                top = stack.pop()
                stack.append(top * num)
                note = f"op cũ '*': {top} * {num} = {top * num}"
            else:
                top = stack.pop()
                stack.append(int(top / num))
                note = f"op cũ '/': cắt về 0 ({top} / {num}) = {int(top / num)}"
            op, num = ch, 0
            note += f"; op mới={ch!r}"
        shown = "(ký tự giả)" if i == len(s) else repr(ch)
        print(f"i={i:>2} ch={shown:>12} | {note:<45} | stack={stack}")
    print("kết quả = sum(stack) =", sum(stack))
    return sum(stack)


trace_stack(" 3+5 / 2 ")

s = ' 3+5 / 2 '
i= 0 ch=         ' ' | bỏ qua khoảng trắng                           | stack=[]
i= 1 ch=         '3' | chữ số -> num=3                               | stack=[]
i= 2 ch=         '+' | op cũ '+': đẩy 3; op mới='+'                  | stack=[3]
i= 3 ch=         '5' | chữ số -> num=5                               | stack=[3]
i= 4 ch=         ' ' | bỏ qua khoảng trắng                           | stack=[3]
i= 5 ch=         '/' | op cũ '+': đẩy 5; op mới='/'                  | stack=[3, 5]
i= 6 ch=         ' ' | bỏ qua khoảng trắng                           | stack=[3, 5]
i= 7 ch=         '2' | chữ số -> num=2                               | stack=[3, 5]
i= 8 ch=         ' ' | bỏ qua khoảng trắng                           | stack=[3, 5]
i= 9 ch= (ký tự giả) | op cũ '/': cắt về 0 (5 / 2) = 2; op mới='+'   | stack=[3, 2]
kết quả = sum(stack) = 5


5

In [13]:
def trace_constant(s):
    print(f"s = {s!r}")
    total, last, num, op, n = 0, 0, 0, "+", len(s)
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if (not ch.isdigit() and ch != " ") or i == n - 1:
            old = (op, num, total, last)
            if op == "+":
                total += last
                last = num
            elif op == "-":
                total += last
                last = -num
            elif op == "*":
                last = last * num
            else:
                last = int(last / num)
            print(f"i={i}: gặp {ch!r}: dùng op cũ {old[0]!r} với num={old[1]} | total {old[2]}->{total}, last {old[3]}->{last}")
            op, num = ch, 0
    print("kết quả = total + last =", total, "+", last, "=", total + last)
    return total + last


trace_constant("3+2*2-4/3")

s = '3+2*2-4/3'
i=1: gặp '+': dùng op cũ '+' với num=3 | total 0->0, last 0->3
i=3: gặp '*': dùng op cũ '+' với num=2 | total 0->3, last 3->2
i=5: gặp '-': dùng op cũ '*' với num=2 | total 3->3, last 2->4
i=7: gặp '/': dùng op cũ '-' với num=4 | total 3->7, last 4->-4
i=8: gặp '3': dùng op cũ '/' với num=3 | total 7->7, last -4->-1
kết quả = total + last = 7 + -1 = 6


6

Bảng tóm tắt Cách 2 (`" 3+5 / 2 "`):

| bước | `num` | `op` cũ | hành động | stack |
|---|---|---|---|---|
| gặp `+` sau `3` | 3 | `+` | đẩy `3` | `[3]` |
| gặp `/` sau `5` | 5 | `+` | đẩy `5` | `[3, 5]` |
| gặp ký tự giả `+` sau `2` | 2 | `/` | `int(5 / 2) = 2` | `[3, 2]` |

`sum([3, 2]) = 5`. Chú ý: phép `/` được xử lý khi gặp dấu **kế tiếp** (hoặc hết chuỗi), không phải khi gặp chính dấu `/`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`O(1)` bộ nhớ). Đúng `class Solution`, đúng `calculate(self, s: str) -> int`, **không** dùng `eval`.

In [14]:
class Solution:
    def calculate(self, s: str) -> int:
        total, last = 0, 0                 # total: tổng các số hạng đã chốt; last: số hạng đang dở
        num = 0
        op = "+"                           # dấu đứng trước num
        n = len(s)
        for i, ch in enumerate(s):
            if ch.isdigit():
                num = num * 10 + int(ch)
            if (not ch.isdigit() and ch != " ") or i == n - 1:
                if op == "+":
                    total += last
                    last = num
                elif op == "-":
                    total += last
                    last = -num
                elif op == "*":
                    last = last * num
                else:
                    last = int(last / num)     # cắt về 0 kể cả khi last âm
                op = ch
                num = 0
        return total + last


sol = Solution()
for s, expected in examples:
    assert sol.calculate(s) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Biểu thức = tổng các số hạng:** vì `*`, `/` ưu tiên hơn `+`, `-` và cùng ưu tiên với nhau (kết hợp trái → phải), mỗi số hạng là một chuỗi nhân/chia từ trái sang phải, và giá trị biểu thức = tổng các số hạng (có dấu).
- **Invariant:** sau khi xử lý toán tử thứ `k`, `total + last` bằng giá trị của phần biểu thức **đã hoàn chỉnh về số** (mọi số trước toán tử đó), với `last` là **số hạng cuối đang dở** chưa bị `+`/`-` tiếp theo chốt. Khi gặp `*`/`/`, số hạng đang dở được cập nhật đúng (nhân/chia tiếp trong cùng số hạng); khi gặp `+`/`-`, số hạng hiện tại được chốt vào `total` và số hạng mới bắt đầu.
- **Dấu đứng trước số:** biến `op` ghi lại toán tử nằm **ngay trước** `num`, nên khi `num` hoàn chỉnh ta biết nó **nối vào đâu**.
- **Chia cắt về 0 đúng:** `last` có thể âm (số hạng `-num`), nên phải dùng `int(last / num)` (cắt về `0`). Ví dụ `1-5/2`: số hạng thứ hai là `-5`, chia `2` ra `-2` (đúng, vì `-(5/2) = -2` với `5/2 = 2`), còn `-5 // 2 = -3` là **sai**.
- **Số cuối:** điều kiện `i == n - 1` (hoặc ký tự giả `+`) đảm bảo số cuối cùng được xử lý dù không có toán tử đứng sau; khoảng trắng ở cuối cũng ổn vì `num` đã gom xong từ trước.
- **Khoảng trắng:** bị bỏ qua hoàn toàn (không đổi `num`, `op`) nên không ảnh hưởng.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (tối đa `3·10^5`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Token + hai lượt | `O(n)` | `O(n)` | hai danh sách trung gian |
| 2. Stack một lượt | `O(n)` | `O(n)` | worst case khi nhiều `+`/`-` |
| 3. `last`/`total` | `O(n)` | `O(1)` | **bản nộp** |
| 4. Văn phạm đệ quy | `O(n)` | `O(n)` | mở rộng cho ngoặc |

Mỗi ký tự được đọc đúng một lần; mỗi phép toán `O(1)` (số nằm trong 32 bit; số lớn hơn của Python vẫn rất nhanh với kích thước này). Không có sự khác biệt "average/worst" về thời gian; chỉ **Space** của Cách 2 phụ thuộc dữ liệu (nhiều `+`/`-` ⇒ stack lớn).

## 12. Bộ test

Ta dùng **`eval` làm oracle kiểm thử** (chỉ để đối chiếu, không trong lời giải): với toán hạng **không âm**, mỗi phép chia trong biểu thức nằm giữa hai số hạng không âm, nên chia cắt về `0` **trùng** với chia sàn `//`; do đó `eval(s.replace("/", "//"))` cho đáp án đúng. Để biểu thức hợp lệ trong Python, ta sinh số **không có số 0 ở đầu** và **không chia cho 0**.

In [15]:
def oracle(s):
    """Đối chiếu bằng eval (chỉ dùng trong test): thay / bằng // vì mọi phép chia có hai vế không âm."""
    return eval(s.strip().replace("/", "//"))


impls = {
    "two_pass": calculate_two_pass,
    "stack": calculate_stack,
    "constant_space": calculate_constant_space,
    "grammar": calculate_grammar,
    "submit": sol.calculate,
}

for s, expected in examples:
    assert oracle(s) == expected
    for key, fn in impls.items():
        assert fn(s) == expected, (key, s, fn(s), expected)
print("PASS ví dụ của đề (mọi cách và oracle eval)")

PASS ví dụ của đề (mọi cách và oracle eval)


### 12.1. Các trường hợp biên
#### Lưu ý: số có số 0 đứng đầu (`"007"`) làm `eval` báo lỗi, nên ta không đưa vào các test dùng oracle.

In [16]:
edge_cases = [
    ("0", 0, "chỉ số 0"),
    ("1", 1, "một chữ số"),
    (" 42 ", 42, "số nhiều chữ số có khoảng trắng hai đầu"),
    ("1-1+1", 1, "cộng trừ trái sang phải"),
    ("2*3*4", 24, "nhân liên tiếp"),
    ("100/10/5", 2, "chia liên tiếp trái sang phải: (100/10)/5"),
    ("7/2*2", 6, "chia rồi nhân: (7/2)*2 = 3*2"),
    ("1-5/2", -1, "số hạng âm trong stack: 1 - (5/2) = 1 - 2 = -1"),
    ("0-5/2", -2, "bắt đầu bằng số hạng âm"),
    ("5/10", 0, "chia ra 0"),
    ("1+2*3-4/2", 5, "đủ bốn phép"),
    ("3*0+5", 5, "nhân với 0"),
    ("2147483647", 2147483647, "số lớn nhất của đề"),
    ("2147483647-2147483647", 0, "hai số lớn nhất trừ nhau"),
    ("  3   +   2  *  2  ", 7, "nhiều khoảng trắng"),
    ("10000*10000/10000", 10000, "trung gian lớn (vẫn nằm trong 32 bit ở LeetCode: 10^8)"),
    ("1-2-3-4", -8, "toàn trừ"),
    ("8/3*3+8-2", 12, "ghép nhiều toán tử: (8/3)*3 + 8 - 2 = 2*3 + 6 = 12"),
]
for s, expected, label in edge_cases:
    assert oracle(s) == expected, (label, oracle(s), expected)
    for key, fn in impls.items():
        assert fn(s) == expected, (key, label, s, fn(s), expected)
    print("PASS", label)

PASS chỉ số 0
PASS một chữ số
PASS số nhiều chữ số có khoảng trắng hai đầu
PASS cộng trừ trái sang phải
PASS nhân liên tiếp
PASS chia liên tiếp trái sang phải: (100/10)/5
PASS chia rồi nhân: (7/2)*2 = 3*2
PASS số hạng âm trong stack: 1 - (5/2) = 1 - 2 = -1
PASS bắt đầu bằng số hạng âm
PASS chia ra 0
PASS đủ bốn phép
PASS nhân với 0
PASS số lớn nhất của đề
PASS hai số lớn nhất trừ nhau
PASS nhiều khoảng trắng
PASS trung gian lớn (vẫn nằm trong 32 bit ở LeetCode: 10^8)
PASS toàn trừ
PASS ghép nhiều toán tử: (8/3)*3 + 8 - 2 = 2*3 + 6 = 12


### 12.2. Tính chất và phản ví dụ

- **Bất biến khi thêm/xóa khoảng trắng** (không chèn giữa các chữ số).
- **`s + "*1"` và `s + "/1"` không đổi giá trị** (nhân/chia số hạng cuối cho `1`).
- **`"0+" + s` không đổi giá trị** (thêm số hạng `0` ở đầu).
- **Phản ví dụ** cho lỗi chia sàn với số âm: `"1-5/2"` có đáp án đúng `-1`, nhưng chia sàn cho `-2`.

In [17]:
def gen_expr(max_terms=6, max_num=999, rng=random, count=None):
    """Sinh biểu thức hợp lệ: số không có số 0 đứng đầu, không chia cho 0, mọi số hạng nằm trong 32 bit."""
    count = count if count is not None else rng.randint(1, max_terms)
    first = rng.randint(0, max_num)
    pieces = [str(first)]
    term = first                                       # giá trị số hạng đang dở (không âm)
    for _ in range(count - 1):
        op = rng.choice("+-*/")
        value = rng.randint(1 if op == "/" else 0, max_num)
        if op == "*" and term * value > 2 ** 31 - 1:   # giữ trong 32 bit như đề đảm bảo
            op = "+"
        if op == "*":
            term *= value
        elif op == "/":
            term //= value
        else:
            term = value
        pieces += [op, str(value)]
    return pieces


def with_spaces(pieces, rng=random):
    out = ""
    for piece in pieces:
        out += " " * rng.randint(0, 2) + piece
    return out + " " * rng.randint(0, 2)


random.seed(227)
for _ in range(1000):
    pieces = gen_expr()
    plain = "".join(pieces)
    spaced = with_spaces(pieces)
    base = sol.calculate(plain)
    assert sol.calculate(spaced) == base                       # khoảng trắng không ảnh hưởng
    assert sol.calculate(plain + "*1") == base                 # nhân số hạng cuối với 1
    assert sol.calculate(plain + "/1") == base                 # chia số hạng cuối cho 1
    assert sol.calculate("0+" + plain) == base                 # thêm số hạng 0 ở đầu
print("PASS tính chất bất biến trên 1000 biểu thức ngẫu nhiên")


def sai_chia_san(s):
    stack, num, op = [], 0, "+"
    for ch in s + "+":
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch != " ":
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(stack.pop() // num)             # SAI: chia sàn với số âm
            op, num = ch, 0
    return sum(stack)


print("phản ví dụ '1-5/2': đúng", sol.calculate("1-5/2"), "| chia sàn //:", sai_chia_san("1-5/2"))
assert sai_chia_san("1-5/2") != sol.calculate("1-5/2")

PASS tính chất bất biến trên 1000 biểu thức ngẫu nhiên
phản ví dụ '1-5/2': đúng -1 | chia sàn //: -2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [18]:
random.seed(4227)
for trial in range(4000):
    pieces = gen_expr(max_terms=random.choice([2, 4, 8, 12]), max_num=random.choice([9, 99, 9999]))
    s = with_spaces(pieces)
    truth = oracle(s)
    for key, fn in impls.items():
        assert fn(s) == truth, (key, s, fn(s), truth)
print("PASS 4000 biểu thức ngẫu nhiên (có khoảng trắng, số 1..4 chữ số): mọi cách khớp oracle eval")

# Biểu thức dài (3 * 10^5 ký tự): không dùng eval (cây cú pháp quá sâu); các cách khớp nhau
random.seed(300_000)
pieces = gen_expr(max_num=99, count=100_000)
big = "".join(pieces)
print("độ dài biểu thức lớn:", len(big))
assert len(big) <= 300_000
ref = calculate_constant_space(big)
for key, fn in impls.items():
    t0 = time.perf_counter()
    value = fn(big)
    dt = (time.perf_counter() - t0) * 1000
    assert value == ref, (key, value, ref)
    print(f"{key:<16} {dt:8.1f} ms")
print("PASS biểu thức ~3*10^5 ký tự: mọi cách khớp nhau")

PASS 4000 biểu thức ngẫu nhiên (có khoảng trắng, số 1..4 chữ số): mọi cách khớp oracle eval


độ dài biểu thức lớn: 290093


two_pass             49.3 ms
stack                25.5 ms


constant_space       41.1 ms
grammar              53.8 ms


submit               41.1 ms
PASS biểu thức ~3*10^5 ký tự: mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính từ trái sang phải, bỏ qua độ ưu tiên
#### Code sai

In [19]:
def sai_bo_uu_tien(s):
    result, num, op = 0, 0, "+"
    for ch in s + "+":
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch != " ":
            if op == "+":
                result += num
            elif op == "-":
                result -= num
            elif op == "*":
                result *= num                              # SAI: nhân cả tổng đã cộng
            else:
                result = int(result / num)
            op, num = ch, 0
    return result


print("đúng:", sol.calculate("3+2*2"), "| sai:", sai_bo_uu_tien("3+2*2"))

đúng: 7 | sai: 10


**Vì sao sai:** `3+2*2` thành `(3+2)*2 = 10`. `*` và `/` chỉ được tác động lên **số hạng đang dở**, không phải toàn bộ tổng. **Code đúng:** nhớ số hạng cuối (`last` hoặc đỉnh stack).

### Lỗi 2: Quên xử lý số cuối cùng
#### Code sai

In [20]:
def sai_quen_so_cuoi(s):
    stack, num, op = [], 0, "+"
    for ch in s:                                           # SAI: không nối thêm '+' / không xử lý i == n-1
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch != " ":
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            op, num = ch, 0
    return sum(stack)


print("đúng:", sol.calculate("3+2*2"), "| sai:", sai_quen_so_cuoi("3+2*2"))
print("đúng:", sol.calculate("42"), "| sai:", sai_quen_so_cuoi("42"))

đúng: 7 | sai: 5
đúng: 42 | sai: 0


**Vì sao sai:** số cuối cùng chỉ được xử lý khi gặp toán tử **sau** nó; không có toán tử nên số `2` cuối (và cả `42`) bị bỏ rơi. **Code đúng:** `for ch in s + "+"` hoặc kiểm tra `i == len(s) - 1`.

### Lỗi 3: Đọc từng chữ số như một số riêng
#### Code sai

In [21]:
def sai_tung_chu_so(s):
    stack, op = [], "+"
    for ch in s + "+":
        if ch.isdigit():
            value = int(ch)                                # SAI: không gom nhiều chữ số
            if op == "+":
                stack.append(value)
            elif op == "-":
                stack.append(-value)
            elif op == "*":
                stack.append(stack.pop() * value)
            else:
                stack.append(int(stack.pop() / value))
        elif ch != " ":
            op = ch
    return sum(stack)


print("đúng:", sol.calculate("12+30"), "| sai:", sai_tung_chu_so("12+30"))

đúng: 42 | sai: 6


**Vì sao sai:** `"12"` bị hiểu thành `1` rồi `2` (cộng thêm một số hạng mới với dấu `+` cũ). **Code đúng:** `num = num * 10 + int(ch)`, chỉ dùng `num` khi gặp toán tử.

### Lỗi 4: Coi khoảng trắng là toán tử
#### Code sai

In [22]:
def sai_khoang_trang(s):
    stack, num, op = [], 0, "+"
    for ch in s + "+":
        if ch.isdigit():
            num = num * 10 + int(ch)
        else:                                              # SAI: khoảng trắng cũng bị coi là toán tử
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / max(num, 1)))
            op, num = ch, 0
    return sum(stack)


for s in ("3+5", "3 +5", " 3+5 / 2 "):
    print(repr(s), "đúng:", sol.calculate(s), "| sai:", sai_khoang_trang(s))

'3+5' đúng: 8 | sai: 8
'3 +5' đúng: 8 | sai: 8
' 3+5 / 2 ' đúng: 5 | sai: 2


**Vì sao sai:** mỗi khoảng trắng bị coi là một "toán tử": nó **chốt `num` hiện tại** (kể cả `num = 0`, đẩy nhầm số `0` vào stack) và đặt `op = ' '`, mà `' '` rơi vào nhánh `else` (nhánh chia). Hai ca đầu (`"3+5"`, `"3 +5"`) "may mắn" cho đúng vì toán tử thật đứng ngay sau bù lại; ca `" 3+5 / 2 "` (cũng là ví dụ 3 của đề) cho `2` thay vì `5`. **Code đúng:** `elif ch != " "` — bỏ qua khoảng trắng hoàn toàn.

### Lỗi 5: Chia sàn `//` với số hạng âm
Xem `sai_chia_san` ở phần test: `"1-5/2"` có số hạng `-5` trong stack; `-5 // 2 = -3` còn đúng phải là `int(-5 / 2) = -2`. **Code đúng:** `int(a / b)` (hoặc `math.trunc`), hoặc giữ **số hạng không âm** cùng dấu riêng như Cách 1.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**tính giá trị biểu thức**" có `+ - * /` mà **không ngoặc** → nghĩ ngay **stack / `last`+`total`**; biểu thức là **tổng các số hạng**.
- Luôn nhớ **dấu đứng trước số**: khởi tạo `op = '+'`, và **xử lý số khi gặp dấu kế tiếp** (hoặc hết chuỗi).
- **Chia cắt về 0:** `int(a / b)` — không dùng `//` khi `a` có thể âm. (Trong C++/Java phép `/` của số nguyên đã cắt về 0.)
- Muốn xử lý số cuối: **nối ký tự giả** `s + "+"` hoặc kiểm tra `i == n - 1`.
- Có **ngoặc** (Basic Calculator I, III) → thêm **đệ quy** (văn phạm) hoặc **stack lưu trạng thái** khi gặp `(`.
- Kiểm thử bằng **`eval`** (đổi `/` thành `//` nếu mọi số không âm) và **sinh biểu thức ngẫu nhiên**: cách rất hiệu quả cho bài parser.
- Giới hạn `n = 3·10^5`: tránh nối chuỗi trong vòng lặp (`O(n²)`); dùng phép toán số `num * 10 + digit`.

## 15. Biến thể của bài

1. **Basic Calculator** (LeetCode 224): chỉ có `+`, `-`, **ngoặc**, có thể có dấu trừ một ngôi.
2. **Basic Calculator III** (772): đủ `+ - * /` và **ngoặc**.
3. **Evaluate Reverse Polish Notation** (150): tính biểu thức hậu tố bằng stack.
4. **Expression Add Operators** (282): chèn toán tử `+ - *` để đạt giá trị mục tiêu (backtracking).
5. **Different Ways to Add Parentheses** (241): mọi cách đặt ngoặc.
6. Thêm **phép lũy thừa** `^` (kết hợp phải) hoặc **số thực**.

In [23]:
def calculate_with_parentheses(s):
    """Biến thể kiểu Calculator III: + - * / ngoặc, dấu trừ một ngôi; / cắt về 0. Văn phạm đệ quy."""
    pos = 0
    n = len(s)

    def skip():
        nonlocal pos
        while pos < n and s[pos] == " ":
            pos += 1

    def factor():                                   # số | ( expr ) | - factor
        nonlocal pos
        skip()
        if s[pos] == "(":
            pos += 1
            value = expr()
            skip()
            pos += 1                                # bỏ ')'
            return value
        if s[pos] == "-":
            pos += 1
            return -factor()
        start = pos
        while pos < n and s[pos].isdigit():
            pos += 1
        return int(s[start:pos])

    def term():
        nonlocal pos
        value = factor()
        while True:
            skip()
            if pos < n and s[pos] in "*/":
                op = s[pos]
                pos += 1
                rhs = factor()
                value = value * rhs if op == "*" else int(value / rhs)
            else:
                return value

    def expr():
        nonlocal pos
        value = term()
        while True:
            skip()
            if pos < n and s[pos] in "+-":
                op = s[pos]
                pos += 1
                rhs = term()
                value = value + rhs if op == "+" else value - rhs
            else:
                return value

    return expr()


tests = ["1 + 1", " 2-1 + 2 ", "(1+(4+5+2)-3)+(6+8)", "2*(5+5*2)/3+(6/2+8)", "-(3+4)*2", "10-(2+3)*(4-1)"]
for t in tests:
    py = t.replace("/", "//")
    print(f"{t!r:<28} -> {calculate_with_parentheses(t):>4} | eval: {eval(py)}")
assert all(calculate_with_parentheses(t) == eval(t.replace("/", "//")) for t in tests if "/" not in t)
assert calculate_with_parentheses("2*(5+5*2)/3+(6/2+8)") == 21

'1 + 1'                      ->    2 | eval: 2
' 2-1 + 2 '                  ->    3 | eval: 3
'(1+(4+5+2)-3)+(6+8)'        ->   23 | eval: 23
'2*(5+5*2)/3+(6/2+8)'        ->   21 | eval: 21
'-(3+4)*2'                   ->  -14 | eval: -14
'10-(2+3)*(4-1)'             ->   -5 | eval: -5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Token + hai lượt | tách token, gộp `*` `/` rồi cộng/trừ | `O(n)` | `O(n)` | dễ hiểu nhất, tách việc rõ ràng |
| 2. Stack một lượt | đẩy số hạng có dấu, `*` `/` tác động lên đỉnh | `O(n)` | `O(n)` | **bản "sách giáo khoa"**, dễ giải thích |
| 3. `last` + `total` | chỉ nhớ số hạng cuối và tổng các số hạng đã chốt | `O(n)` | `O(1)` | **bản nộp tối ưu** |
| 4. Văn phạm đệ quy | `expr → term`, `term → số` | `O(n)` | `O(n)` | mở rộng sang ngoặc, toán tử mới |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `int(-7 / 2)`; (b) `-7 // 2`; (c) `"12+3".isdigit()` và `"12".isdigit()`; (d) `Solution().calculate("2*3+4")`.

**Bài 2 (tính bằng tay).** Tính `"14-3*2/4+5"` theo từng số hạng và theo bảng trạng thái của Cách 3 (`total`, `last` sau mỗi toán tử).

**Bài 3 (điền TODO).** Hoàn thiện `calc_todo(s)` theo Cách 2 (stack).

In [24]:
def calc_todo(s):
    stack = []
    num = 0
    op = "+"
    # TODO 1: duyệt ch trong s + "+"
    # TODO 2: nếu ch là chữ số: num = num * 10 + int(ch)
    # TODO 3: nếu ch không phải khoảng trắng (là toán tử): xử lý theo op cũ
    #         '+': đẩy num; '-': đẩy -num; '*': đỉnh *= num; '/': đỉnh = int(đỉnh / num)
    # TODO 4: cập nhật op = ch, num = 0
    # TODO 5: trả sum(stack)
    pass


# Tự làm xong rồi mới mở test:
# for s, expected in examples:
#     assert calc_todo(s) == expected

**Bài 4 (tự viết lại).** Viết `calc_constant(s)` theo Cách 3 (`last`/`total`), **không nhìn lại** code mẫu; kiểm tra với `oracle` trên biểu thức ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Viết `calc_plus_minus_parens(s)` tính biểu thức chỉ có `+`, `-`, số và **ngoặc** (Basic Calculator I) bằng **stack lưu `(tổng_trước_ngoặc, dấu_trước_ngoặc)`**; thử `"(1+(4+5+2)-3)+(6+8)"` → `23`. Gợi ý: gặp `(` đẩy `(result, sign)`; gặp `)` lấy ra và `result = old_result + sign * result`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `int()` bỏ phần thập phân (cắt về 0); `//` làm tròn xuống; `isdigit` chỉ đúng khi **mọi** ký tự là chữ số. Bài 2: chia thành số hạng `+14`, `-(3*2/4)`, `+5`. Bài 5: mỗi ngoặc là một biểu thức con, cần nhớ trạng thái bên ngoài.
- **Mức 2 (vừa):** Bài 1: `-3`, `-4`, `False` và `True`, `10`. Bài 2: `3*2 = 6`, `6/4 = 1` (cắt về 0), số hạng `-1`; tổng `14 - 1 + 5 = 18`. Bài 3: đúng khung Cách 2. Bài 4: `total += last; last = ±num` cho `+`/`-`; sửa `last` cho `*`/`/`.
- **Mức 3 (sâu):** Bài 2 bảng Cách 3: gặp `-` (sau 14): `total=0,last=14`; gặp `*` (sau 3, op cũ `-`): `total=14,last=-3`; gặp `/` (sau 2, op cũ `*`): `last=-6`; gặp `+` (sau 4, op cũ `/`): `last=int(-6/4)=-1`; hết chuỗi (sau 5, op cũ `+`): `total=14-1=13,last=5` ⇒ `18`. Bài 5: `result, sign, num` và `stack`; `+` hoặc `-` chốt `result += sign * num` rồi cập nhật `sign`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert int(-7 / 2) == -3 and -7 // 2 == -4
assert "12+3".isdigit() is False and "12".isdigit() is True
assert Solution().calculate("2*3+4") == 10

# Bài 2
assert Solution().calculate("14-3*2/4+5") == 18 == oracle("14-3*2/4+5")


# Bài 3
def calc_todo_ans(s):
    stack = []
    num = 0
    op = "+"
    for ch in s + "+":
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch != " ":
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            op, num = ch, 0
    return sum(stack)


for s, expected in examples:
    assert calc_todo_ans(s) == expected


# Bài 4
def calc_constant(s):
    total, last, num, op = 0, 0, 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if (not ch.isdigit() and ch != " ") or i == len(s) - 1:
            if op == "+":
                total, last = total + last, num
            elif op == "-":
                total, last = total + last, -num
            elif op == "*":
                last *= num
            else:
                last = int(last / num)
            op, num = ch, 0
    return total + last


random.seed(8)
for _ in range(500):
    s = with_spaces(gen_expr())
    assert calc_constant(s) == oracle(s)


# Bài 5
def calc_plus_minus_parens(s):
    stack = []                          # lưu (kết quả trước ngoặc, dấu trước ngoặc)
    result, sign, num = 0, 1, 0
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num = 0
            sign = 1 if ch == "+" else -1
        elif ch == "(":
            stack.append((result, sign))
            result, sign, num = 0, 1, 0
        elif ch == ")":
            result += sign * num
            num = 0
            old_result, old_sign = stack.pop()
            result = old_result + old_sign * result
    return result + sign * num


assert calc_plus_minus_parens("(1+(4+5+2)-3)+(6+8)") == 23
assert calc_plus_minus_parens("1-(2+3)") == -4
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (`last` + `total`, `O(1)` bộ nhớ)

```python
class Solution:
    def calculate(self, s: str) -> int:
        total, last, num, op = 0, 0, 0, "+"
        for i, ch in enumerate(s):
            if ch.isdigit():
                num = num * 10 + int(ch)
            if (not ch.isdigit() and ch != " ") or i == len(s) - 1:
                if op == "+":
                    total += last
                    last = num
                elif op == "-":
                    total += last
                    last = -num
                elif op == "*":
                    last *= num
                else:
                    last = int(last / num)
                op, num = ch, 0
        return total + last
```

**Checklist**

- [ ] Khởi tạo `op = '+'`, `num = 0`; **xử lý số khi gặp toán tử kế tiếp** hoặc ở ký tự cuối?
- [ ] Gom chữ số bằng `num = num * 10 + int(ch)` (không đọc từng chữ số riêng)?
- [ ] **Bỏ qua khoảng trắng** (`ch != " "`) và không coi nó là toán tử?
- [ ] `*`/`/` chỉ tác động lên **số hạng đang dở** (`last` / đỉnh stack), không phải cả tổng?
- [ ] Chia **cắt về 0** bằng `int(a / b)`, không dùng `//` với số hạng âm?
- [ ] Không dùng `eval` trong lời giải nộp (chỉ dùng khi test)?
- [ ] Đã thử `"0"`, `" 42 "`, `"1-5/2"`, `"7/2*2"`, nhiều khoảng trắng, số `2147483647`?
- [ ] Đã đối chiếu với `eval(s.replace("/", "//"))` trên biểu thức ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def calculate(self, s: str) -> int`?